In [18]:
import json
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, UMT5Config, UMT5ForConditionalGeneration, Seq2SeqTrainingArguments,Seq2SeqTrainer, EarlyStoppingCallback 
import torch
from datasets import Dataset, DatasetDict
import transformers, dataclasses
from tqdm.auto import tqdm
import difflib

---
## Load Dataset

In [5]:
def load_jsonl(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        records = [json.loads(line) for line in f]
    return pd.DataFrame(records)

train_df = load_jsonl("../data/processed/train.jsonl")
val_df = load_jsonl("../data/processed/val.jsonl")
test_real_df = load_jsonl("../data/processed/test_real.jsonl")
test_regression_df = load_jsonl("../data/processed/test_regression.jsonl")

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "validation": Dataset.from_pandas(val_df),
    "test_real": Dataset.from_pandas(test_real_df),
    "test_regression": Dataset.from_pandas(test_regression_df),
})

In [6]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 168326 entries, 0 to 168325
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   corrupted  168326 non-null  str  
 1   clean      168326 non-null  str  
 2   source     168326 non-null  str  
dtypes: str(3)
memory usage: 85.1 MB


---
## Tokenizer and model

In [7]:
MODEL_NAME = "google/umt5-small"

In [8]:
MAX_LENGTH = 128 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    model_inputs = tokenizer(
        examples["corrupted"],
        max_length=MAX_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=examples["clean"],
        max_length=MAX_LENGTH,
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [9]:
config = UMT5Config.from_pretrained(MODEL_NAME)
config.tie_word_embeddings = False          # umt5 untied — сохраняем обученный lm_head при загрузке
model = UMT5ForConditionalGeneration.from_pretrained(MODEL_NAME, config=config)

In [10]:
def unify_columns(ds):
    rename_map = {}
    if "input" in ds.column_names:
        rename_map["input"] = "corrupted"
    if "target" in ds.column_names:
        rename_map["target"] = "clean"
    return ds.rename_columns(rename_map) if rename_map else ds

dataset = DatasetDict({split: unify_columns(ds) for split, ds in dataset.items()})

In [11]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
)

In [12]:
tokenized_datasets = DatasetDict({
    split: ds.map(preprocess, batched=True, remove_columns=ds.column_names)
    for split, ds in dataset.items()
})

print(tokenized_datasets)
print(tokenized_datasets["train"][0])

Map: 100%|██████████| 105/105 [00:00<00:00, 17498.49 examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 168326
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2400
    })
    test_real: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 120
    })
    test_regression: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 105
    })
})
{'input_ids': [297, 110099, 20944, 273, 284, 280, 1174, 90123, 1109, 105042, 463, 16970, 463, 1174, 292, 914, 296, 150299, 274, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [297, 110099, 20944, 273, 284, 280, 1174, 90123, 1109, 76954, 10056, 54745, 73180, 274, 1]}


---

## Training

In [22]:
training_args = Seq2SeqTrainingArguments(
    output_dir="../models/umt5-gec-small",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,      
    bf16=True,
    optim="adafactor",
    num_train_epochs=1,
    learning_rate=3e-4,
    warmup_steps=500,
    lr_scheduler_type="linear",
    group_by_length=True,               
    eval_strategy="steps", eval_steps=1000,
    save_strategy="steps", save_steps=1000,
    save_total_limit=2, logging_steps=50,
    predict_with_generate=False,        
    generation_max_length=MAX_LENGTH,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

In [23]:
model.config.use_cache = False

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()


trainer.save_model("../models/umt5-gec-small/best")
tokenizer.save_pretrained("../models/umt5-gec-small/best")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


Step,Training Loss,Validation Loss
1000,0.630000,0.388591
2000,0.487600,0.284214
3000,0.412900,0.231090
4000,0.389200,0.213168
5000,0.369000,0.206780


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


('../models/umt5-gec-fixed/best\\tokenizer_config.json',
 '../models/umt5-gec-fixed/best\\special_tokens_map.json',
 '../models/umt5-gec-fixed/best\\spiece.model',
 '../models/umt5-gec-fixed/best\\added_tokens.json',
 '../models/umt5-gec-fixed/best\\tokenizer.json')

## Evaluation

In [16]:
device = model.device
model.eval()

def generate_corrections(texts, batch_size=16, max_new_tokens=128, num_beams=5):
    preds = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                            truncation=True, max_length=128).to(device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                num_beams=num_beams,
                length_penalty=1.0,
                early_stopping=True,
            )
        preds.extend(tokenizer.batch_decode(out, skip_special_tokens=True))
    return preds

# --- test_real ---
real_preds = generate_corrections(test_real_df["corrupted"].tolist())
test_real_df["prediction"] = real_preds

# --- test_regression ---
test_regression_df = test_regression_df.rename(columns={"input": "corrupted", "target": "clean"})
reg_preds = generate_corrections(test_regression_df["corrupted"].tolist())
test_regression_df["prediction"] = reg_preds

100%|██████████| 7/7 [02:04<00:00, 17.82s/it]


In [19]:
def get_edits(a: str, b: str):
    """Return set of word-level edits when transforming a -> b"""
    a_tok, b_tok = a.split(), b.split()
    sm = difflib.SequenceMatcher(None, a_tok, b_tok)
    edits = set()
    for tag, i1, i2, j1, j2 in sm.get_opcodes():
        if tag != "equal":
            edits.add((tag, i1, i2, tuple(a_tok[i1:i2]), tuple(b_tok[j1:j2])))
    return edits

def edit_prf(corrupted, clean, pred):
    """Compare gold edits (corrupted->clean) vs predicted edits (corrupted->pred)"""
    gold_edits = get_edits(corrupted, clean)
    pred_edits = get_edits(corrupted, pred)
    tp = len(gold_edits & pred_edits)
    fp = len(pred_edits - gold_edits)
    fn = len(gold_edits - pred_edits)
    return tp, fp, fn

def f_beta(tp, fp, fn, beta=0.5):
    # F0.5 weights precision higher than recall — standard for GEC
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = (1 + beta**2) * p * r / (beta**2 * p + r) if (p + r) else 0.0
    return p, r, f

# --- test_real: precision/recall/F0.5 over word-level edits ---
tp = fp = fn = 0
for _, row in test_real_df.iterrows():
    t, f, n = edit_prf(row["corrupted"], row["clean"], row["prediction"])
    tp += t; fp += f; fn += n

p, r, f05 = f_beta(tp, fp, fn)
print(f"test_real  P={p:.3f}  R={r:.3f}  F0.5={f05:.3f}")

# --- test_regression: model should not touch already-clean text ---
unchanged = (test_regression_df["corrupted"] == test_regression_df["prediction"]).mean()
print(f"test_regression unchanged rate: {unchanged:.3f}")  # should be close to 1.0

# --- baseline (identity), to know the floor ---
tp = fp = fn = 0
for _, row in test_real_df.iterrows():
    t, f, n = edit_prf(row["corrupted"], row["clean"], row["corrupted"])
    tp += t; fp += f; fn += n
p0, r0, f0 = f_beta(tp, fp, fn)
print(f"baseline (identity)  P={p0:.3f}  R={r0:.3f}  F0.5={f0:.3f}")

test_real  P=0.000  R=0.000  F0.5=0.000
test_regression unchanged rate: 0.000
baseline (identity)  P=0.000  R=0.000  F0.5=0.000


In [20]:
for i in range(5):
    row = test_real_df.iloc[i]
    print("CORRUPTED :", row["corrupted"])
    print("CLEAN     :", row["clean"])
    print("PREDICTION:", row["prediction"])
    print("-" * 40)

CORRUPTED : Грамшидің пайымдауыңа, әр адамның iс-әрекети, минимум денгейде болса да, техникалык кабілетти, яғни шығармашылык интелектуалдық функцияны кажет етет.
CLEAN     : Грамшидің пайымдауынша, әр адамның іc-әрекеті, минимум деңгейде болса да, техникалық қабілетті, яғни шығармашылық интеллектуалдық функцияны қажет етеді.
PREDICTION: ,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
----------------------------------------
CORRUPTED : Сондықтан жастарды жумыс наргында жумысмен камтамасыз ету ары олардын белсенділігін, бәсекеге кабілеттiлігін жоғары денгейде ұстап отыру важды.
CLEAN     : Сондықтан жастарды жұмыс нарығында жұмыспен қамтамасыз ету әрі олардың белсенділігін, бәсекеге қабілеттілігін жоғары деңгейде ұстап отыру маңызды.
PREDICTION: , жумыс, жумыс, жумыс, жумыс, жумыс, жумыс, жумыс,, жумыс, жумыс, жумыс,,,,,,,,,,, жумыс, жумыс,,,, жумыс, жумыс,,,,,,,, жумыс,,,,,,,,,,,,, жумыс, жумыс, жумыс, жумыс,,
----------------------------------------
CORRUPTED : Зерттеулерг